# 📥 인스모바일 MVNO 정산 - 원시데이터 로드 및 전처리

## 개요
이 노트북은 `csv` 폴더 내의 정산 원시데이터 파일들을 로드하여 분석 준비를 수행하고, 전처리된 데이터를 `output` 폴더에 CSV 파일로 저장합니다.

## 주요 기능
1. **원시데이터 파일 탐색**: `YYYYMM_SS001344_ENTR_BY_STACC_PTN_INS_001.csv` 형식의 파일 자동 탐색
2. **데이터 로드**: 각 파일을 판다스 데이터프레임으로 로드 (데이터프레임명: `LGU_YYYYMM`)
3. **유효시작월 컬럼 추가**: 유효시작일자 컬럼을 기준으로 유효시작월(YYYYMM) 컬럼 자동 추가
4. **수익 컬럼 추가**: 수납금액에서 도매대가합계(1~24+82)를 차감하여 수익 컬럼 자동 추가
   - 계산식: 수익 = 수납금액 - 도매대가합계(1~24+82)
5. **MVNO상품구분 컬럼 추가**: MVNO상품명 컬럼 값을 기준으로 상품 구분 분류
   - `[INS][24개월]` 포함 → "24개월 요금제"
   - `[INS][평생할인]` 포함 → "평생 요금제"
   - 나머지 → "기타 요금제"
6. **선불 항목 필터링**: MVNO상품명에 "선불"이 포함된 항목 제외
7. **피봇 테이블 생성**: 유효시작월(열)과 MVNO상품명(행) 별 수익 컬럼 합계 금액 테이블 생성
   - `LGU_YYYYMM_PIVOT` 데이터프레임으로 저장
8. **피봇 테이블 요약 정보**: 생성된 피봇 테이블의 상세 요약 정보 출력
   - 기본 정보, 상품명별/유효시작월별 수익 통계, 전체 통계 및 분석
9. **데이터프레임 요약 정보 출력**: 각 데이터프레임의 기본 통계 및 정보 출력
   - 컬럼 정보 (표 형태 - 전체 표시)
   - 선후불 구분 컬럼 분석
10. **MVNO상품명 컬럼 상세 분석**: 별도 섹션으로 구분된 5단계 분석 프로세스
   - 1단계: 컬럼 자동 감지
   - 2단계: 기본 통계 정보 수집
   - 3단계: 유형값별 통계 계산 (전체 유형값)
   - 4단계: 표 형태로 출력 (전체 유형값 표시)
   - 5단계: 추가 통계 정보 (최다/최소 상품명)
11. **CSV 파일 저장**: 전처리된 데이터프레임 및 피봇 테이블을 `output` 폴더에 CSV 파일로 저장
   - 원본 데이터프레임: `output/LGU_YYYYMM_processed.csv`
   - 피봇 테이블: `output/LGU_YYYYMM_pivot.csv`

## 입력 파일 형식
- 파일 경로: `csv/YYYYMM_SS001344_ENTR_BY_STACC_PTN_INS_001.csv`
  - 예: `csv/202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv`
  - 예: `csv/202510_SS001344_ENTR_BY_STACC_PTN_INS_001.csv`
- 데이터프레임명: `LGU_YYYYMM` 형식
  - 예: `LGU_202507` (2025년 7월 데이터)
  - 예: `LGU_202510` (2025년 10월 데이터)

## 출력 정보
- 각 데이터프레임의 행 수, 열 수
- 파일 크기 및 메모리 사용량
- **유효시작월 컬럼**: 유효시작일자 기준 YYYYMM 형식 컬럼 자동 추가
- **수익 컬럼**: 수납금액 - 도매대가합계(1~24+82) 계산 결과 및 통계 정보
- **MVNO상품구분 컬럼**: MVNO상품명 기준 상품 구분 분류 결과 및 통계 정보
- **선불 항목 필터링**: MVNO상품명에 "선불" 포함 항목 제외 결과 및 통계 정보
- **피봇 테이블**: 유효시작월 × MVNO상품명 기준 수익 합계 피봇 테이블
- **피봇 테이블 요약 정보**: 상세 통계 및 분석 정보
- 컬럼 목록 (표 형태 - 전체 컬럼 표시)
- 기본 통계 정보
- **선후불 구분 컬럼**: 유형값 및 각 유형별 개수와 비율
- **MVNO상품명 컬럼**: 유형값 및 각 유형별 개수와 비율 (전체 표시)

## 출력 파일
전처리된 데이터는 `output` 폴더에 CSV 파일로 저장됩니다:
- **원본 데이터프레임**: `output/LGU_YYYYMM_processed.csv` (전처리된 원본 데이터)
- **피봇 테이블**: `output/LGU_YYYYMM_pivot.csv` (유효시작월 × MVNO상품명 피봇 테이블)


## 1️⃣ 환경 설정 및 라이브러리 로드

필요한 라이브러리를 import하고 pandas 출력 옵션을 설정합니다.


In [47]:
# 필요한 라이브러리 import
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import re
from datetime import datetime
warnings.filterwarnings('ignore')

# Pandas 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)  # 모든 행 표시
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)  # 컬럼 너비 제한
pd.set_option('display.float_format', '{:.2f}'.format)

print("\n" + "="*70)
print("📥 인스모바일 MVNO 정산 - 원시데이터 로드")
print("="*70)
print("✅ 라이브러리 로드 완료!")



📥 인스모바일 MVNO 정산 - 원시데이터 로드
✅ 라이브러리 로드 완료!


## 2️⃣ 원시데이터 파일 탐색

`csv` 폴더에서 `YYYYMM_SS001344_ENTR_BY_STACC_PTN_INS_001.csv` 형식의 파일을 자동으로 탐색합니다.


In [50]:
# CSV 디렉토리 설정
BASE_DIR = Path(".")
CSV_DIR = BASE_DIR / "csv"

print("\n" + "="*70)
print("🔍 원시데이터 파일 탐색")
print("="*70)
print(f"📂 검색 디렉토리: {CSV_DIR.absolute()}")

# 파일 패턴: YYYYMM_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
file_pattern = re.compile(r'^(\d{6})_SS001344_ENTR_BY_STACC_PTN_INS_001\.csv$')

# CSV 디렉토리에서 매칭되는 파일 찾기
matching_files = []
if CSV_DIR.exists():
    for file_path in CSV_DIR.glob('*_SS001344_ENTR_BY_STACC_PTN_INS_001.csv'):
        match = file_pattern.match(file_path.name)
        if match:
            yyyymm = match.group(1)
            matching_files.append({
                'file_path': file_path,
                'filename': file_path.name,
                'yyyymm': yyyymm,
                'year': yyyymm[:4],
                'month': yyyymm[4:]
            })

# 날짜순으로 정렬
matching_files.sort(key=lambda x: x['yyyymm'])

print(f"\n📋 발견된 파일: {len(matching_files)}개")
if len(matching_files) > 0:
    print("\n파일 목록:")
    for i, file_info in enumerate(matching_files, 1):
        file_size = file_info['file_path'].stat().st_size / 1024 / 1024
        print(f"  {i}. {file_info['filename']} ({file_info['year']}년 {file_info['month']}월, {file_size:.2f} MB)")
else:
    print("\n⚠️ 매칭되는 파일을 찾을 수 없습니다.")
    print(f"   검색 패턴: YYYYMM_SS001344_ENTR_BY_STACC_PTN_INS_001.csv")
    print(f"   디렉토리: {CSV_DIR.absolute()}")



🔍 원시데이터 파일 탐색
📂 검색 디렉토리: d:\Dev\Python_Data_Agent\csv

📋 발견된 파일: 4개

파일 목록:
  1. 202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv (2025년 07월, 121.97 MB)
  2. 202508_SS001344_ENTR_BY_STACC_PTN_INS_001.csv (2025년 08월, 242.89 MB)
  3. 202509_SS001344_ENTR_BY_STACC_PTN_INS_001.csv (2025년 09월, 233.50 MB)
  4. 202510_SS001344_ENTR_BY_STACC_PTN_INS_001.csv (2025년 10월, 248.70 MB)


## 3️⃣ 원시데이터 로드

탐색된 CSV 파일들을 판다스 데이터프레임으로 로드합니다. 다양한 인코딩을 자동으로 시도하여 파일을 읽습니다.


In [51]:
def load_csv_with_encoding(file_path, encodings=['utf-8', 'cp949', 'euc-kr', 'utf-8-sig']):
    """다양한 인코딩으로 CSV 파일 로드 시도"""
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
            return df, encoding
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ⚠️ {encoding} 시도 중 오류: {str(e)[:100]}")
            continue
    raise ValueError(f"❌ 파일 로드 실패 - 지원되는 인코딩이 없습니다: {file_path}")

print("\n" + "="*70)
print("📥 원시데이터 로드 시작")
print("="*70)

# 전역 변수로 데이터프레임 저장을 위한 딕셔너리
dataframes = {}

# 각 파일을 로드하여 데이터프레임으로 저장
for file_info in matching_files:
    file_path = file_info['file_path']
    yyyymm = file_info['yyyymm']
    df_name = f"LGU_{yyyymm}"
    
    print(f"\n📂 [{yyyymm}] {file_info['filename']}")
    
    try:
        # 파일 로드
        df, encoding = load_csv_with_encoding(file_path)
        
        # 전역 네임스페이스에 데이터프레임 저장
        globals()[df_name] = df
        dataframes[yyyymm] = {
            'df': df,
            'name': df_name,
            'file_path': file_path,
            'encoding': encoding,
            'yyyymm': yyyymm
        }
        
        print(f"  ✅ 로드 성공 (인코딩: {encoding})")
        print(f"  📊 데이터 크기: {len(df):,}행 × {len(df.columns)}열")
        print(f"  💾 데이터프레임명: {df_name}")
        
    except Exception as e:
        print(f"  ❌ 로드 실패: {str(e)[:200]}")
        continue

print("\n" + "="*70)
print(f"✅ 원시데이터 로드 완료! (총 {len(dataframes)}개 파일)")
print("="*70)

# 로드된 데이터프레임 목록 출력
if len(dataframes) > 0:
    print("\n📋 로드된 데이터프레임:")
    for yyyymm in sorted(dataframes.keys()):
        df_info = dataframes[yyyymm]
        print(f"  - {df_info['name']}: {len(df_info['df']):,}행 × {len(df_info['df'].columns)}열")



📥 원시데이터 로드 시작

📂 [202507] 202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  ✅ 로드 성공 (인코딩: cp949)
  📊 데이터 크기: 287,638행 × 111열
  💾 데이터프레임명: LGU_202507

📂 [202508] 202508_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  ✅ 로드 성공 (인코딩: cp949)
  📊 데이터 크기: 265,215행 × 111열
  💾 데이터프레임명: LGU_202508

📂 [202509] 202509_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  ✅ 로드 성공 (인코딩: cp949)
  📊 데이터 크기: 254,760행 × 111열
  💾 데이터프레임명: LGU_202509

📂 [202510] 202510_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  ✅ 로드 성공 (인코딩: cp949)
  📊 데이터 크기: 271,789행 × 111열
  💾 데이터프레임명: LGU_202510

✅ 원시데이터 로드 완료! (총 4개 파일)

📋 로드된 데이터프레임:
  - LGU_202507: 287,638행 × 111열
  - LGU_202508: 265,215행 × 111열
  - LGU_202509: 254,760행 × 111열
  - LGU_202510: 271,789행 × 111열


## 3-1️⃣ 유효시작월 컬럼 추가

각 데이터프레임의 유효시작일자 컬럼을 기준으로 유효시작월(YYYYMM) 컬럼을 자동으로 추가합니다.

**처리 과정**:
1. 유효시작일자 컬럼 자동 감지
2. datetime 타입으로 변환
3. YYYYMM 형식의 유효시작월 컬럼 생성 및 추가
4. 통계 정보 출력


In [52]:
print("\n" + "="*70)
print("📅 유효시작월 컬럼 추가")
print("="*70)

# 각 데이터프레임에 유효시작월 컬럼 추가
for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print("-" * 70)
    
    # ========================================
    # 1단계: 유효시작일자 컬럼 찾기
    # ========================================
    valid_start_cols = ['유효시작일자', '유효시작일', '시작일자', '유효시작일자']
    valid_start_col = None
    
    for col in valid_start_cols:
        if col in df.columns:
            valid_start_col = col
            break
    
    if valid_start_col is None:
        print(f"  ⚠️ 유효시작일자 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(valid_start_cols))}")
        continue
    
    print(f"  ✅ 유효시작일자 컬럼 발견: '{valid_start_col}'")
    
    # ========================================
    # 2단계: 유효시작일자를 datetime으로 변환
    # ========================================
    print(f"\n[2단계] 유효시작일자 데이터 타입 변환")
    print("-" * 70)
    
    # 기존 데이터 타입 확인
    original_dtype = df[valid_start_col].dtype
    print(f"  - 원본 데이터 타입: {original_dtype}")
    
    # datetime으로 변환
    df[valid_start_col] = pd.to_datetime(df[valid_start_col], errors='coerce')
    
    # 변환 결과 확인
    converted_count = df[valid_start_col].notna().sum()
    total_count = len(df)
    print(f"  - 변환 성공: {converted_count:,}행 / {total_count:,}행")
    
    if converted_count < total_count:
        failed_count = total_count - converted_count
        print(f"  - 변환 실패: {failed_count:,}행 (결측값 또는 잘못된 형식)")
    
    # ========================================
    # 3단계: 유효시작월 컬럼 생성 (YYYYMM 형식)
    # ========================================
    print(f"\n[3단계] 유효시작월 컬럼 생성")
    print("-" * 70)
    
    # 유효시작일자 컬럼의 위치 찾기
    col_index = df.columns.get_loc(valid_start_col)
    print(f"  - 유효시작일자 컬럼 위치: {col_index + 1}번째")
    
    # 유효시작월 컬럼 생성 (YYYYMM 형식)
    # 예: 2025-10-15 → 202510
    valid_start_month = df[valid_start_col].dt.to_period('M').astype(str).str.replace('-', '')
    
    # 유효시작일자 컬럼 옆에 유효시작월 컬럼 추가
    df.insert(
        col_index + 1,
        '유효시작월',
        valid_start_month
    )
    
    print(f"  ✅ '유효시작월' 컬럼 추가 완료")
    print(f"     위치: '{valid_start_col}' 컬럼 바로 옆 ({col_index + 2}번째)")
    
    # ========================================
    # 4단계: 유효시작월 통계 정보
    # ========================================
    print(f"\n[4단계] 유효시작월 통계 정보")
    print("-" * 70)
    
    # 유효시작월 값 분포
    month_counts = df['유효시작월'].value_counts().sort_index()
    total_months = len(month_counts)
    
    print(f"  - 총 유효시작월 종류: {total_months}개")
    print(f"  - 유효시작월 분포 (상위 10개):")
    
    for i, (month, count) in enumerate(month_counts.head(10).items(), 1):
        percentage = (count / total_count) * 100
        if pd.isna(month):
            print(f"    {i:2d}. (결측값): {count:,}건 ({percentage:.2f}%)")
        else:
            # YYYYMM 형식을 YYYY년 MM월로 표시
            year = month[:4]
            mon = month[4:]
            print(f"    {i:2d}. {year}년 {mon}월 ({month}): {count:,}건 ({percentage:.2f}%)")
    
    if total_months > 10:
        print(f"    ... 외 {total_months - 10}개 월")
    
    # 결측값 정보
    missing_count = df['유효시작월'].isnull().sum()
    if missing_count > 0:
        missing_percentage = (missing_count / total_count) * 100
        print(f"\n  - 결측값: {missing_count:,}건 ({missing_percentage:.2f}%)")
    
    # 최소/최대 유효시작월
    valid_months = df['유효시작월'].dropna()
    if len(valid_months) > 0:
        min_month = valid_months.min()
        max_month = valid_months.max()
        min_year = min_month[:4] if pd.notna(min_month) else "N/A"
        min_mon = min_month[4:] if pd.notna(min_month) else "N/A"
        max_year = max_month[:4] if pd.notna(max_month) else "N/A"
        max_mon = max_month[4:] if pd.notna(max_month) else "N/A"
        
        print(f"\n  - 최소 유효시작월: {min_year}년 {min_mon}월 ({min_month})")
        print(f"  - 최대 유효시작월: {max_year}년 {max_mon}월 ({max_month})")
    
    # 데이터프레임 업데이트
    dataframes[yyyymm]['df'] = df
    globals()[df_name] = df
    
    print(f"\n  ✅ {df_name}에 '유효시작월' 컬럼 추가 완료!")
    print(f"     현재 컬럼 수: {len(df.columns)}개 (이전: {len(df.columns) - 1}개)")

print("\n" + "="*70)
print("✅ 모든 데이터프레임에 유효시작월 컬럼 추가 완료!")
print("="*70)



📅 유효시작월 컬럼 추가

📋 LGU_202507 (2025년 07월)
----------------------------------------------------------------------
  ✅ 유효시작일자 컬럼 발견: '유효시작일자'

[2단계] 유효시작일자 데이터 타입 변환
----------------------------------------------------------------------
  - 원본 데이터 타입: object
  - 변환 성공: 287,479행 / 287,638행
  - 변환 실패: 159행 (결측값 또는 잘못된 형식)

[3단계] 유효시작월 컬럼 생성
----------------------------------------------------------------------
  - 유효시작일자 컬럼 위치: 9번째
  ✅ '유효시작월' 컬럼 추가 완료
     위치: '유효시작일자' 컬럼 바로 옆 (10번째)

[4단계] 유효시작월 통계 정보
----------------------------------------------------------------------
  - 총 유효시작월 종류: 54개
  - 유효시작월 분포 (상위 10개):
     1. 2021년 03월 (202103): 4건 (0.00%)
     2. 2021년 04월 (202104): 25건 (0.01%)
     3. 2021년 05월 (202105): 16건 (0.01%)
     4. 2021년 06월 (202106): 19건 (0.01%)
     5. 2021년 07월 (202107): 15건 (0.01%)
     6. 2021년 08월 (202108): 42건 (0.01%)
     7. 2021년 09월 (202109): 29건 (0.01%)
     8. 2021년 10월 (202110): 19건 (0.01%)
     9. 2021년 11월 (202111): 68건 (0.02%)
    10. 2021년 12월 (2021

## 3-2️⃣ 수익 컬럼 추가

수납금액에서 도매대가합계(1~24+82)를 차감하여 수익 컬럼을 자동으로 추가합니다.

**계산식**: 수익 = 수납금액 - 도매대가합계(1~24+82)

**처리 과정**:
1. 수납금액 및 도매대가합계 컬럼 자동 감지
2. 데이터 타입 확인 및 숫자형 변환
3. 수익 컬럼 계산 및 추가
4. 수익 통계 정보 출력 (합계, 평균, 분포 등)


In [53]:
print("\n" + "="*70)
print("💰 수익 컬럼 추가")
print("="*70)

# 각 데이터프레임에 수익 컬럼 추가
for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print("-" * 70)
    
    # ========================================
    # 1단계: 필요한 컬럼 찾기
    # ========================================
    print(f"[1단계] 필요한 컬럼 찾기")
    print("-" * 70)
    
    # 수납금액 컬럼 찾기
    payment_cols = ['수납금액', '수납급액', '수납액']
    payment_col = None
    
    for col in payment_cols:
        if col in df.columns:
            payment_col = col
            break
    
    if payment_col is None:
        print(f"  ⚠️ 수납금액 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(payment_cols))}")
        continue
    
    print(f"  ✅ 수납금액 컬럼 발견: '{payment_col}'")
    
    # 도매대가합계(1~24+82) 컬럼 찾기
    wholesale_cols = ['도매대가합계(1~24+82)', '도매대가합계', '도매대가']
    wholesale_col = None
    
    for col in wholesale_cols:
        if col in df.columns:
            wholesale_col = col
            break
    
    if wholesale_col is None:
        print(f"  ⚠️ 도매대가합계(1~24+82) 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(wholesale_cols))}")
        continue
    
    print(f"  ✅ 도매대가합계 컬럼 발견: '{wholesale_col}'")
    
    # ========================================
    # 2단계: 데이터 타입 확인 및 변환
    # ========================================
    print(f"\n[2단계] 데이터 타입 확인 및 변환")
    print("-" * 70)
    
    # 수납금액 데이터 타입 확인
    payment_dtype = df[payment_col].dtype
    print(f"  - 수납금액 데이터 타입: {payment_dtype}")
    
    # 도매대가합계 데이터 타입 확인
    wholesale_dtype = df[wholesale_col].dtype
    print(f"  - 도매대가합계 데이터 타입: {wholesale_dtype}")
    
    # 숫자형으로 변환 (필요한 경우)
    payment_values = pd.to_numeric(df[payment_col], errors='coerce')
    wholesale_values = pd.to_numeric(df[wholesale_col], errors='coerce')
    
    # 변환 결과 확인
    payment_valid = payment_values.notna().sum()
    wholesale_valid = wholesale_values.notna().sum()
    total_count = len(df)
    
    print(f"  - 수납금액 유효값: {payment_valid:,}행 / {total_count:,}행")
    print(f"  - 도매대가합계 유효값: {wholesale_valid:,}행 / {total_count:,}행")
    
    # ========================================
    # 3단계: 수익 컬럼 계산 및 추가
    # ========================================
    print(f"\n[3단계] 수익 컬럼 계산 및 추가")
    print("-" * 70)
    
    # 수익 계산: 수납금액 - 도매대가합계(1~24+82)
    revenue = payment_values - wholesale_values
    
    # 도매대가합계 컬럼의 위치 찾기
    wholesale_col_index = df.columns.get_loc(wholesale_col)
    print(f"  - 도매대가합계 컬럼 위치: {wholesale_col_index + 1}번째")
    
    # 도매대가합계 컬럼 옆에 수익 컬럼 추가
    df.insert(
        wholesale_col_index + 1,
        '수익',
        revenue
    )
    
    print(f"  ✅ '수익' 컬럼 추가 완료")
    print(f"     위치: '{wholesale_col}' 컬럼 바로 옆 ({wholesale_col_index + 2}번째)")
    print(f"     계산식: 수익 = {payment_col} - {wholesale_col}")
    
    # ========================================
    # 4단계: 수익 통계 정보
    # ========================================
    print(f"\n[4단계] 수익 통계 정보")
    print("-" * 70)
    
    # 기본 통계
    revenue_valid = df['수익'].notna().sum()
    revenue_missing = df['수익'].isnull().sum()
    
    print(f"  - 유효값: {revenue_valid:,}행")
    print(f"  - 결측값: {revenue_missing:,}행 ({revenue_missing/total_count*100:.2f}%)")
    
    if revenue_valid > 0:
        revenue_stats = df['수익'].describe()
        
        print(f"\n  - 수익 통계:")
        print(f"    • 합계: {df['수익'].sum():,.2f}원")
        print(f"    • 평균: {df['수익'].mean():,.2f}원")
        print(f"    • 중앙값: {df['수익'].median():,.2f}원")
        print(f"    • 최소값: {df['수익'].min():,.2f}원")
        print(f"    • 최대값: {df['수익'].max():,.2f}원")
        print(f"    • 표준편차: {df['수익'].std():,.2f}원")
        
        # 양수/음수/0 분포
        positive_count = (df['수익'] > 0).sum()
        negative_count = (df['수익'] < 0).sum()
        zero_count = (df['수익'] == 0).sum()
        
        print(f"\n  - 수익 분포:")
        print(f"    • 양수: {positive_count:,}건 ({positive_count/revenue_valid*100:.2f}%)")
        print(f"    • 음수: {negative_count:,}건 ({negative_count/revenue_valid*100:.2f}%)")
        print(f"    • 0: {zero_count:,}건 ({zero_count/revenue_valid*100:.2f}%)")
        
        # 수납금액과 도매대가합계 비교
        print(f"\n  - 수납금액 vs 도매대가합계 비교:")
        print(f"    • 수납금액 합계: {payment_values.sum():,.2f}원")
        print(f"    • 도매대가합계 합계: {wholesale_values.sum():,.2f}원")
        print(f"    • 수익 합계: {df['수익'].sum():,.2f}원")
        
        if payment_values.sum() != 0:
            revenue_ratio = (df['수익'].sum() / payment_values.sum()) * 100
            print(f"    • 수익률: {revenue_ratio:.2f}%")
    
    # 데이터프레임 업데이트
    dataframes[yyyymm]['df'] = df
    globals()[df_name] = df
    
    print(f"\n  ✅ {df_name}에 '수익' 컬럼 추가 완료!")
    print(f"     현재 컬럼 수: {len(df.columns)}개 (이전: {len(df.columns) - 1}개)")

print("\n" + "="*70)
print("✅ 모든 데이터프레임에 수익 컬럼 추가 완료!")
print("="*70)



💰 수익 컬럼 추가

📋 LGU_202507 (2025년 07월)
----------------------------------------------------------------------
[1단계] 필요한 컬럼 찾기
----------------------------------------------------------------------
  ✅ 수납금액 컬럼 발견: '수납금액'
  ✅ 도매대가합계 컬럼 발견: '도매대가합계(1~24+82)'

[2단계] 데이터 타입 확인 및 변환
----------------------------------------------------------------------
  - 수납금액 데이터 타입: float64
  - 도매대가합계 데이터 타입: float64
  - 수납금액 유효값: 287,638행 / 287,638행
  - 도매대가합계 유효값: 287,638행 / 287,638행

[3단계] 수익 컬럼 계산 및 추가
----------------------------------------------------------------------
  - 도매대가합계 컬럼 위치: 21번째
  ✅ '수익' 컬럼 추가 완료
     위치: '도매대가합계(1~24+82)' 컬럼 바로 옆 (22번째)
     계산식: 수익 = 수납금액 - 도매대가합계(1~24+82)

[4단계] 수익 통계 정보
----------------------------------------------------------------------
  - 유효값: 287,638행
  - 결측값: 0행 (0.00%)

  - 수익 통계:
    • 합계: -609,727,815.24원
    • 평균: -2,119.77원
    • 중앙값: -330.00원
    • 최소값: -478,802.50원
    • 최대값: 755,936.13원
    • 표준편차: 8,645.42원

  - 수익 분포:
    • 양수: 42,607건 (14.81%)
    

## 3-3️⃣ MVNO상품구분 컬럼 추가

MVNO상품명 컬럼 값을 기준으로 상품 구분을 자동 분류하여 MVNO상품구분 컬럼을 추가합니다.

**분류 기준**:
- `[INS][24개월]` 포함 → "24개월 요금제"
- `[INS][평생할인]` 포함 → "평생 요금제"
- 나머지 → "기타 요금제"

**처리 과정**:
1. MVNO상품명 컬럼 자동 감지
2. 상품명 패턴 분석 및 분류
3. MVNO상품구분 컬럼 생성 및 추가
4. 분류별 통계 정보 및 샘플 상품명 출력


In [54]:
print("\n" + "="*70)
print("🏷️ MVNO상품구분 컬럼 추가")
print("="*70)

# 각 데이터프레임에 MVNO상품구분 컬럼 추가
for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print("-" * 70)
    
    # ========================================
    # 1단계: MVNO상품명 컬럼 찾기
    # ========================================
    print(f"[1단계] MVNO상품명 컬럼 찾기")
    print("-" * 70)
    
    # 가능한 컬럼명 목록 (우선순위 순서)
    product_cols = ['MVNO상품명', '상품명', '요금제명', '개통요금제명', '현재요금제명']
    product_col = None
    
    # 컬럼명 자동 검색
    for col in product_cols:
        if col in df.columns:
            product_col = col
            print(f"  ✅ 컬럼 발견: '{col}'")
            break
    
    # 컬럼을 찾지 못한 경우
    if product_col is None:
        print(f"  ⚠️ MVNO상품명 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(product_cols))}")
        continue
    
    # ========================================
    # 2단계: MVNO상품구분 컬럼 생성
    # ========================================
    print(f"\n[2단계] MVNO상품구분 컬럼 생성")
    print("-" * 70)
    
    # MVNO상품명 컬럼의 위치 찾기
    product_col_index = df.columns.get_loc(product_col)
    print(f"  - MVNO상품명 컬럼 위치: {product_col_index + 1}번째")
    
    # MVNO상품구분 컬럼 생성
    # 조건에 따라 분류
    def classify_product(product_name):
        """MVNO상품명을 기준으로 상품구분 분류"""
        if pd.isna(product_name):
            return "기타 요금제"
        
        product_str = str(product_name)
        
        # [INS][24개월] 포함 확인
        if '[24개월]' in product_str:
            return "24개월 요금제"
        
        # [INS][평생할인] 포함 확인
        if '[평생할인]' in product_str:
            return "평생 요금제"
        
        # 나머지
        return "기타 요금제"
    
    # MVNO상품구분 컬럼 생성
    product_classification = df[product_col].apply(classify_product)
    
    # MVNO상품명 컬럼 옆에 MVNO상품구분 컬럼 추가
    df.insert(
        product_col_index + 1,
        'MVNO상품구분',
        product_classification
    )
    
    print(f"  ✅ 'MVNO상품구분' 컬럼 추가 완료")
    print(f"     위치: '{product_col}' 컬럼 바로 옆 ({product_col_index + 2}번째)")
    
    # ========================================
    # 3단계: MVNO상품구분 통계 정보
    # ========================================
    print(f"\n[3단계] MVNO상품구분 통계 정보")
    print("-" * 70)
    
    # MVNO상품구분 값 분포
    classification_counts = df['MVNO상품구분'].value_counts()
    total_count = len(df)
    
    print(f"  - 총 분류 종류: {len(classification_counts)}개")
    print(f"  - MVNO상품구분 분포:")
    
    for classification, count in classification_counts.items():
        percentage = (count / total_count) * 100
        print(f"    • {classification}: {count:,}건 ({percentage:.2f}%)")
    
    # 각 분류별 샘플 상품명 출력
    print(f"\n  - 분류별 샘플 상품명:")
    
    for classification in classification_counts.index:
        sample_products = df[df['MVNO상품구분'] == classification][product_col].dropna().head(3)
        print(f"\n    [{classification}]")
        if len(sample_products) > 0:
            for i, product in enumerate(sample_products, 1):
                product_str = str(product)
                if len(product_str) > 60:
                    product_str = product_str[:60] + "..."
                print(f"      {i}. {product_str}")
        else:
            print(f"      (샘플 없음)")
    
    # 데이터프레임 업데이트
    dataframes[yyyymm]['df'] = df
    globals()[df_name] = df
    
    print(f"\n  ✅ {df_name}에 'MVNO상품구분' 컬럼 추가 완료!")
    print(f"     현재 컬럼 수: {len(df.columns)}개 (이전: {len(df.columns) - 1}개)")

print("\n" + "="*70)
print("✅ 모든 데이터프레임에 MVNO상품구분 컬럼 추가 완료!")
print("="*70)



🏷️ MVNO상품구분 컬럼 추가

📋 LGU_202507 (2025년 07월)
----------------------------------------------------------------------
[1단계] MVNO상품명 컬럼 찾기
----------------------------------------------------------------------
  ✅ 컬럼 발견: 'MVNO상품명'

[2단계] MVNO상품구분 컬럼 생성
----------------------------------------------------------------------
  - MVNO상품명 컬럼 위치: 5번째
  ✅ 'MVNO상품구분' 컬럼 추가 완료
     위치: 'MVNO상품명' 컬럼 바로 옆 (6번째)

[3단계] MVNO상품구분 통계 정보
----------------------------------------------------------------------
  - 총 분류 종류: 3개
  - MVNO상품구분 분포:
    • 기타 요금제: 234,283건 (81.45%)
    • 평생 요금제: 30,015건 (10.43%)
    • 24개월 요금제: 23,340건 (8.11%)

  - 분류별 샘플 상품명:

    [기타 요금제]
      1. [INS]인스 선불정액 300MB
      2. [INS]인스 선불정액 300MB
      3. [INS]인스 선불정액 300MB

    [평생 요금제]
      1. [INS][평생할인]인스 유심 스트롱 11GB+
      2. [INS][평생할인]인스 유심 스트롱 11GB+
      3. [INS][평생할인]인스 유심 스트롱 11GB+

    [24개월 요금제]
      1. [INS][24개월] 인스 유심 스트롱 11GB+
      2. [INS][24개월] 인스 유심 스트롱 11GB+
      3. [INS][24개월] 인스 유심 스트롱 11GB+

  ✅ LGU_20250

## 3-4️⃣ 선불 항목 필터링

MVNO상품명 컬럼에 "선불"이 포함된 항목을 제외하여 데이터를 필터링합니다.

**처리 과정**:
1. MVNO상품명 컬럼 자동 감지
2. 필터링 전 통계 확인
3. "선불" 포함 항목 제외
4. 필터링 후 통계 및 상위 상품명 출력


In [55]:
print("\n" + "="*70)
print("🔍 선불 항목 필터링")
print("="*70)

# 각 데이터프레임에서 선불 항목 제외
for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print("-" * 70)
    
    # ========================================
    # 1단계: MVNO상품명 컬럼 찾기
    # ========================================
    print(f"[1단계] MVNO상품명 컬럼 찾기")
    print("-" * 70)
    
    # 가능한 컬럼명 목록 (우선순위 순서)
    product_cols = ['MVNO상품명', '상품명', '요금제명', '개통요금제명', '현재요금제명']
    product_col = None
    
    # 컬럼명 자동 검색
    for col in product_cols:
        if col in df.columns:
            product_col = col
            print(f"  ✅ 컬럼 발견: '{col}'")
            break
    
    # 컬럼을 찾지 못한 경우
    if product_col is None:
        print(f"  ⚠️ MVNO상품명 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(product_cols))}")
        continue
    
    # ========================================
    # 2단계: 필터링 전 통계
    # ========================================
    print(f"\n[2단계] 필터링 전 통계")
    print("-" * 70)
    
    total_count_before = len(df)
    
    # "선불" 포함 여부 확인
    # MVNO상품명에 "선불"이 포함된 행 찾기
    prepaid_mask = df[product_col].astype(str).str.contains('선불', na=False)
    prepaid_count = prepaid_mask.sum()
    
    print(f"  - 필터링 전 총 행 수: {total_count_before:,}행")
    print(f"  - '선불' 포함 항목: {prepaid_count:,}행 ({prepaid_count/total_count_before*100:.2f}%)")
    print(f"  - 필터링 후 예상 행 수: {total_count_before - prepaid_count:,}행")
    
    # ========================================
    # 3단계: 선불 항목 제외 필터링
    # ========================================
    print(f"\n[3단계] 선불 항목 제외 필터링")
    print("-" * 70)
    
    # "선불"이 포함되지 않은 행만 선택
    df_filtered = df[~prepaid_mask].copy()
    
    total_count_after = len(df_filtered)
    removed_count = total_count_before - total_count_after
    
    print(f"  ✅ 필터링 완료")
    print(f"     제외된 행 수: {removed_count:,}행")
    print(f"     남은 행 수: {total_count_after:,}행")
    print(f"     제외 비율: {removed_count/total_count_before*100:.2f}%")
    
    # ========================================
    # 4단계: 필터링 후 통계
    # ========================================
    print(f"\n[4단계] 필터링 후 통계")
    print("-" * 70)
    
    # MVNO상품명 분포 확인
    if total_count_after > 0:
        product_counts = df_filtered[product_col].value_counts()
        print(f"  - 남은 상품명 종류: {len(product_counts)}개")
        print(f"  - 상위 5개 상품명:")
        for i, (product, count) in enumerate(product_counts.head(5).items(), 1):
            percentage = (count / total_count_after) * 100
            product_str = str(product)
            if len(product_str) > 60:
                product_str = product_str[:60] + "..."
            print(f"    {i}. {product_str}: {count:,}건 ({percentage:.2f}%)")
    
    # 데이터프레임 업데이트
    dataframes[yyyymm]['df'] = df_filtered
    globals()[df_name] = df_filtered
    
    print(f"\n  ✅ {df_name} 필터링 완료!")
    print(f"     필터링 전: {total_count_before:,}행")
    print(f"     필터링 후: {total_count_after:,}행")
    print(f"     감소: {removed_count:,}행 ({removed_count/total_count_before*100:.2f}%)")

print("\n" + "="*70)
print("✅ 모든 데이터프레임 선불 항목 필터링 완료!")
print("="*70)



🔍 선불 항목 필터링

📋 LGU_202507 (2025년 07월)
----------------------------------------------------------------------
[1단계] MVNO상품명 컬럼 찾기
----------------------------------------------------------------------
  ✅ 컬럼 발견: 'MVNO상품명'

[2단계] 필터링 전 통계
----------------------------------------------------------------------
  - 필터링 전 총 행 수: 287,638행
  - '선불' 포함 항목: 116,999행 (40.68%)
  - 필터링 후 예상 행 수: 170,639행

[3단계] 선불 항목 제외 필터링
----------------------------------------------------------------------
  ✅ 필터링 완료
     제외된 행 수: 116,999행
     남은 행 수: 170,639행
     제외 비율: 40.68%

[4단계] 필터링 후 통계
----------------------------------------------------------------------
  - 남은 상품명 종류: 133개
  - 상위 5개 상품명:
    1. 번호이동 정산상품: 57,354건 (33.61%)
    2. [INS][24개월] 인스 유심 스트롱 11GB+: 14,432건 (8.46%)
    3. [INS][평생할인]데이터 안심 10GB+(USIM): 11,360건 (6.66%)
    4. [INS]인스 유심 스트롱 11G+: 8,226건 (4.82%)
    5. [INS] 인스 유심 스트롱 15GB+: 8,199건 (4.80%)

  ✅ LGU_202507 필터링 완료!
     필터링 전: 287,638행
     필터링 후: 170,639행
     감소: 116,999행 (40

## 3-5️⃣ 피봇 테이블 생성

유효시작월(열)과 MVNO상품명(행)을 기준으로 수익 컬럼 합계 금액 피봇 테이블을 생성합니다.

**피봇 테이블 구조**:
- **행(Index)**: MVNO상품명
- **열(Columns)**: 유효시작월
- **값(Values)**: 수익 컬럼 합계 금액

**생성 데이터프레임**: `LGU_YYYYMM_PIVOT` 형식 (예: `LGU_202507_PIVOT`)

**처리 과정**:
1. 필요한 컬럼 확인 (유효시작월, MVNO상품명, 수익)
2. 행 수 기준 피봇 테이블 생성 (참고용)
3. 수납금액 합계 기준 피봇 테이블 생성 (참고용)
4. 수익 합계 기준 피봇 테이블 생성 (최종)
5. 피봇 테이블 저장 및 요약 정보 출력


In [56]:
print("\n" + "="*70)
print("📊 피봇 테이블 생성")
print("="*70)

# 각 데이터프레임에 대해 피봇 테이블 생성
pivot_dataframes = {}

for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    pivot_df_name = f"{df_name}_PIVOT"
    
    print(f"\n📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월) → {pivot_df_name}")
    print("-" * 70)
    
    # ========================================
    # 1단계: 필요한 컬럼 확인
    # ========================================
    print(f"[1단계] 필요한 컬럼 확인")
    print("-" * 70)
    
    # 유효시작월 컬럼 확인
    valid_start_month_col = '유효시작월'
    if valid_start_month_col not in df.columns:
        print(f"  ⚠️ '유효시작월' 컬럼을 찾을 수 없습니다.")
        continue
    
    print(f"  ✅ 유효시작월 컬럼: '{valid_start_month_col}'")
    
    # MVNO상품명 컬럼 확인
    product_cols = ['MVNO상품명', '상품명', '요금제명', '개통요금제명', '현재요금제명']
    product_col = None
    
    for col in product_cols:
        if col in df.columns:
            product_col = col
            break
    
    if product_col is None:
        print(f"  ⚠️ MVNO상품명 컬럼을 찾을 수 없습니다.")
        continue
    
    print(f"  ✅ MVNO상품명 컬럼: '{product_col}'")
    
    # 수납금액 컬럼 확인
    payment_cols = ['수납금액', '수납급액', '수납액']
    payment_col = None
    
    for col in payment_cols:
        if col in df.columns:
            payment_col = col
            break
    
    if payment_col:
        print(f"  ✅ 수납금액 컬럼: '{payment_col}'")
    else:
        print(f"  ⚠️ 수납금액 컬럼을 찾을 수 없습니다. (행 수만 계산)")
    
    # 수익 컬럼 확인
    revenue_col = '수익'
    if revenue_col in df.columns:
        print(f"  ✅ 수익 컬럼: '{revenue_col}'")
    else:
        print(f"  ⚠️ 수익 컬럼을 찾을 수 없습니다. (수익 합계 제외)")
    
    # ========================================
    # 2단계: 피봇 테이블 생성 (행 수)
    # ========================================
    print(f"\n[2단계] 피봇 테이블 생성 (행 수)")
    print("-" * 70)
    
    # 행 수 기준 피봇 테이블 생성
    # 가입번호나 다른 고유 컬럼을 사용하여 정확한 행 수 계산
    unique_cols = ['가입번호', '정산년월', '파트너ID']
    value_col = None
    for col in unique_cols:
        if col in df.columns:
            value_col = col
            break
    
    if value_col:
        # 고유 컬럼을 사용하여 count
        pivot_count = pd.pivot_table(
            df,
            index=product_col,
            columns=valid_start_month_col,
            values=value_col,
            aggfunc='count',
            fill_value=0
        )
    else:
        # value_col이 없으면 size를 사용하여 행 수 계산
        pivot_count = df.groupby([product_col, valid_start_month_col]).size().unstack(fill_value=0)
    
    print(f"  ✅ 행 수 기준 피봇 테이블 생성 완료")
    print(f"     행 수: {len(pivot_count)}개 (상품명)")
    print(f"     열 수: {len(pivot_count.columns)}개 (유효시작월)")
    
    # ========================================
    # 3단계: 피봇 테이블 생성 (수납금액 합계)
    # ========================================
    pivot_payment = None
    if payment_col:
        print(f"\n[3단계] 피봇 테이블 생성 (수납금액 합계)")
        print("-" * 70)
        
        pivot_payment = pd.pivot_table(
            df,
            index=product_col,
            columns=valid_start_month_col,
            values=payment_col,
            aggfunc='sum',
            fill_value=0
        )
        
        print(f"  ✅ 수납금액 합계 기준 피봇 테이블 생성 완료")
    
    # ========================================
    # 4단계: 피봇 테이블 생성 (수익 합계)
    # ========================================
    pivot_revenue = None
    if revenue_col in df.columns:
        print(f"\n[4단계] 피봇 테이블 생성 (수익 합계)")
        print("-" * 70)
        
        pivot_revenue = pd.pivot_table(
            df,
            index=product_col,
            columns=valid_start_month_col,
            values=revenue_col,
            aggfunc='sum',
            fill_value=0
        )
        
        print(f"  ✅ 수익 합계 기준 피봇 테이블 생성 완료")
    
    # ========================================
    # 5단계: 최종 피봇 테이블 생성 및 저장
    # ========================================
    print(f"\n[5단계] 최종 피봇 테이블 생성 및 저장")
    print("-" * 70)
    
    # 수익 합계 기준 피봇 테이블을 기본으로 사용
    if pivot_revenue is not None:
        pivot_final = pivot_revenue.copy()
    elif pivot_payment is not None:
        # 수익이 없으면 수납금액 합계 사용
        pivot_final = pivot_payment.copy()
    else:
        # 수납금액도 없으면 행 수 기준 사용
        pivot_final = pivot_count.copy()
    pivot_final.columns.name = '유효시작월'
    pivot_final.index.name = 'MVNO상품명'
    
    # ========================================
    # MVNO상품구분 컬럼 추가
    # ========================================
    # MVNO상품구분 컬럼 확인
    product_classification_col = 'MVNO상품구분'
    if product_classification_col in df.columns:
        # 원본 데이터프레임에서 MVNO상품명과 MVNO상품구분의 매핑 생성
        # 각 상품명에 대해 첫 번째로 나타나는 상품구분 값을 사용
        product_classification_map = df.groupby(product_col)[product_classification_col].first()
        
        # 피봇 테이블의 인덱스(상품명)에 대응하는 상품구분 매핑
        pivot_classification = pivot_final.index.map(product_classification_map)
        
        # MVNO상품구분 컬럼을 첫 번째 컬럼으로 추가
        pivot_final.insert(0, product_classification_col, pivot_classification)
        
        # 상품구분 통계 출력 (Series로 변환 후 value_counts)
        classification_counts = pd.Series(pivot_classification).value_counts().to_dict()
        print(f"  ✅ MVNO상품구분 컬럼 추가 완료")
        print(f"     상품구분 종류: {classification_counts}")
    else:
        print(f"  ⚠️ MVNO상품구분 컬럼을 찾을 수 없습니다. 컬럼 추가 생략")
    
    # 전역 변수로 저장
    globals()[pivot_df_name] = pivot_final
    pivot_dataframes[yyyymm] = {
        'df': pivot_final,
        'name': pivot_df_name,
        'count': pivot_count,
        'payment': pivot_payment,
        'revenue': pivot_revenue,
        'yyyymm': yyyymm
    }
    
    print(f"  ✅ {pivot_df_name} 생성 완료!")
    print(f"     데이터프레임 크기: {len(pivot_final)}행 × {len(pivot_final.columns)}열")
    
    # ========================================
    # 6단계: 피봇 테이블 요약 정보 출력
    # ========================================
    print(f"\n[6단계] 피봇 테이블 요약 정보")
    print("-" * 70)
    
    # 기본 통계
    print(f"  - 피봇 테이블 크기: {len(pivot_final)}행 × {len(pivot_final.columns)}열")
    print(f"  - 총 셀 수: {len(pivot_final) * len(pivot_final.columns):,}개")
    print(f"  - 0이 아닌 셀 수: {(pivot_final != 0).sum().sum():,}개")
    
    # 행 합계 (상품명별 총합) - 금액 기준
    # MVNO상품구분 컬럼이 있으면 제외하고 숫자 컬럼만 선택
    numeric_cols = pivot_final.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        row_sums = pivot_final[numeric_cols].sum(axis=1).sort_values(ascending=False)
    else:
        row_sums = pivot_final.sum(axis=1).sort_values(ascending=False)
    print(f"\n  - 상품명별 금액 총합 (상위 5개):")
    for i, (product, total) in enumerate(row_sums.head(5).items(), 1):
        product_str = str(product)
        if len(product_str) > 50:
            product_str = product_str[:50] + "..."
        print(f"    {i}. {product_str}: {total:,.2f}원")
    
    # 열 합계 (유효시작월별 총합) - 금액 기준
    # MVNO상품구분 컬럼이 있으면 제외하고 숫자 컬럼만 선택
    numeric_cols = pivot_final.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        col_sums = pivot_final[numeric_cols].sum(axis=0).sort_index()
    else:
        col_sums = pivot_final.sum(axis=0).sort_index()
    print(f"\n  - 유효시작월별 금액 총합:")
    for month, total in col_sums.items():
        if pd.notna(month):
            year = str(month)[:4] if len(str(month)) >= 4 else "N/A"
            mon = str(month)[4:] if len(str(month)) >= 6 else "N/A"
            print(f"    {year}년 {mon}월 ({month}): {total:,.2f}원")
    
    # 전체 합계 - 금액 기준
    # MVNO상품구분 컬럼이 있으면 제외하고 숫자 컬럼만 선택
    numeric_cols = pivot_final.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        total_sum = pivot_final[numeric_cols].sum().sum()
    else:
        total_sum = pivot_final.sum().sum()
    print(f"\n  - 전체 금액 총합: {total_sum:,.2f}원")
    
    # 행 수 피봇 테이블 요약 (참고용)
    if pivot_count is not None:
        print(f"\n  - 행 수 피봇 테이블 (참고용):")
        count_total = pivot_count.sum().sum()
        print(f"    전체 행 수 합계: {count_total:,.0f}건")
    
    # 수납금액 피봇 테이블 요약 (참고용)
    if pivot_payment is not None:
        print(f"\n  - 수납금액 피봇 테이블 (참고용):")
        payment_total = pivot_payment.sum().sum()
        print(f"    전체 수납금액 합계: {payment_total:,.2f}원")
    
    print(f"\n  ✅ {pivot_df_name} 요약 정보 출력 완료!")

print("\n" + "="*70)
print("✅ 모든 데이터프레임 피봇 테이블 생성 완료!")
print("="*70)

# 생성된 피봇 테이블 목록 출력
if len(pivot_dataframes) > 0:
    print("\n📋 생성된 피봇 테이블:")
    for yyyymm in sorted(pivot_dataframes.keys()):
        pivot_info = pivot_dataframes[yyyymm]
        print(f"  - {pivot_info['name']}: {len(pivot_info['df'])}행 × {len(pivot_info['df'].columns)}열")



📊 피봇 테이블 생성

📋 LGU_202507 (2025년 07월) → LGU_202507_PIVOT
----------------------------------------------------------------------
[1단계] 필요한 컬럼 확인
----------------------------------------------------------------------
  ✅ 유효시작월 컬럼: '유효시작월'
  ✅ MVNO상품명 컬럼: 'MVNO상품명'
  ✅ 수납금액 컬럼: '수납금액'
  ✅ 수익 컬럼: '수익'

[2단계] 피봇 테이블 생성 (행 수)
----------------------------------------------------------------------
  ✅ 행 수 기준 피봇 테이블 생성 완료
     행 수: 133개 (상품명)
     열 수: 52개 (유효시작월)

[3단계] 피봇 테이블 생성 (수납금액 합계)
----------------------------------------------------------------------
  ✅ 수납금액 합계 기준 피봇 테이블 생성 완료

[4단계] 피봇 테이블 생성 (수익 합계)
----------------------------------------------------------------------
  ✅ 수익 합계 기준 피봇 테이블 생성 완료

[5단계] 최종 피봇 테이블 생성 및 저장
----------------------------------------------------------------------
  ✅ MVNO상품구분 컬럼 추가 완료
     상품구분 종류: {'기타 요금제': 110, '평생 요금제': 14, '24개월 요금제': 9}
  ✅ LGU_202507_PIVOT 생성 완료!
     데이터프레임 크기: 133행 × 53열

[6단계] 피봇 테이블 요약 정보
---------------------------------------

## 3-6️⃣ 피봇 테이블 요약 정보

생성된 `LGU_YYYYMM_PIVOT` 데이터프레임의 상세 요약 정보를 출력합니다.

**요약 정보 내용**:
1. **기본 정보**: 피봇 테이블 크기, 메모리 사용량, 0이 아닌 셀 수
2. **상품명별 수익 통계**: 상위/하위 상품명, 평균/중앙값/표준편차
3. **유효시작월별 수익 통계**: 상위/하위 월, 평균/중앙값/표준편차
4. **전체 통계 및 분석**: 양수/음수/0 분포, 수익 합계 분포, 셀 값 통계
5. **샘플 데이터**: 상위 5개 상품명 × 상위 5개 유효시작월 교차표


In [57]:
print("\n" + "="*70)
print("📊 피봇 테이블 요약 정보")
print("="*70)

# 각 피봇 테이블에 대한 상세 요약 정보 출력
for yyyymm in sorted(pivot_dataframes.keys()):
    pivot_info = pivot_dataframes[yyyymm]
    pivot_df = pivot_info['df']
    pivot_df_name = pivot_info['name']
    
    print(f"\n{'='*70}")
    print(f"📋 {pivot_df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print(f"{'='*70}")
    
    # ========================================
    # 1단계: 기본 정보
    # ========================================
    print(f"\n[1단계] 기본 정보")
    print("-" * 70)
    
    print(f"  - 데이터프레임명: {pivot_df_name}")
    print(f"  - 행 수: {len(pivot_df):,}행 (MVNO상품명)")
    print(f"  - 열 수: {len(pivot_df.columns):,}열 (유효시작월)")
    print(f"  - 총 셀 수: {len(pivot_df) * len(pivot_df.columns):,}개")
    
    # 메모리 사용량
    memory_usage = pivot_df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"  - 메모리 사용량: {memory_usage:.2f} MB")
    
    # 0이 아닌 셀 수 (숫자 컬럼만 선택)
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_pivot = pivot_df[numeric_cols]
        non_zero_count = (numeric_pivot != 0).sum().sum()
        zero_count = (numeric_pivot == 0).sum().sum()
        total_cells = len(numeric_pivot) * len(numeric_pivot.columns)
        non_zero_percentage = (non_zero_count / total_cells) * 100 if total_cells > 0 else 0
    else:
        non_zero_count = (pivot_df != 0).sum().sum()
        zero_count = (pivot_df == 0).sum().sum()
        total_cells = len(pivot_df) * len(pivot_df.columns)
        non_zero_percentage = (non_zero_count / total_cells) * 100 if total_cells > 0 else 0
    print(f"  - 0이 아닌 셀 수: {non_zero_count:,}개 ({non_zero_percentage:.2f}%)")
    print(f"  - 0인 셀 수: {zero_count:,}개 ({100 - non_zero_percentage:.2f}%)")
    
    # ========================================
    # 2단계: 상품명별 수익 통계
    # ========================================
    print(f"\n[2단계] 상품명별 수익 통계")
    print("-" * 70)
    
    # 행 합계 (상품명별 총합) - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        row_sums = pivot_df[numeric_cols].sum(axis=1).sort_values(ascending=False)
    else:
        row_sums = pivot_df.sum(axis=1).sort_values(ascending=False)
    
    print(f"  - 상위 10개 상품명 (수익 기준):")
    for i, (product, total) in enumerate(row_sums.head(10).items(), 1):
        percentage = (total / row_sums.sum()) * 100 if row_sums.sum() != 0 else 0
        product_str = str(product)
        if len(product_str) > 50:
            product_str = product_str[:50] + "..."
        print(f"    {i:2d}. {product_str}: {total:,.2f}원 ({percentage:.2f}%)")
    
    if len(row_sums) > 10:
        print(f"\n  - 하위 5개 상품명 (수익 기준):")
        for i, (product, total) in enumerate(row_sums.tail(5).items(), 1):
            percentage = (total / row_sums.sum()) * 100 if row_sums.sum() != 0 else 0
            product_str = str(product)
            if len(product_str) > 50:
                product_str = product_str[:50] + "..."
            print(f"    {i:2d}. {product_str}: {total:,.2f}원 ({percentage:.2f}%)")
    
    # 상품명별 통계
    print(f"\n  - 상품명별 수익 통계:")
    print(f"    • 평균: {row_sums.mean():,.2f}원")
    print(f"    • 중앙값: {row_sums.median():,.2f}원")
    print(f"    • 최대값: {row_sums.max():,.2f}원")
    print(f"    • 최소값: {row_sums.min():,.2f}원")
    print(f"    • 표준편차: {row_sums.std():,.2f}원")
    
    # ========================================
    # 3단계: 유효시작월별 수익 통계
    # ========================================
    print(f"\n[3단계] 유효시작월별 수익 통계")
    print("-" * 70)
    
    # 열 합계 (유효시작월별 총합) - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        col_sums = pivot_df[numeric_cols].sum(axis=0).sort_values(ascending=False)
    else:
        col_sums = pivot_df.sum(axis=0).sort_values(ascending=False)
    
    print(f"  - 상위 10개 유효시작월 (수익 기준):")
    for i, (month, total) in enumerate(col_sums.head(10).items(), 1):
        percentage = (total / col_sums.sum()) * 100 if col_sums.sum() != 0 else 0
        if pd.notna(month):
            year = str(month)[:4] if len(str(month)) >= 4 else "N/A"
            mon = str(month)[4:] if len(str(month)) >= 6 else "N/A"
            print(f"    {i:2d}. {year}년 {mon}월 ({month}): {total:,.2f}원 ({percentage:.2f}%)")
        else:
            print(f"    {i:2d}. (결측값): {total:,.2f}원 ({percentage:.2f}%)")
    
    if len(col_sums) > 10:
        print(f"\n  - 하위 5개 유효시작월 (수익 기준):")
        for i, (month, total) in enumerate(col_sums.tail(5).items(), 1):
            percentage = (total / col_sums.sum()) * 100 if col_sums.sum() != 0 else 0
            if pd.notna(month):
                year = str(month)[:4] if len(str(month)) >= 4 else "N/A"
                mon = str(month)[4:] if len(str(month)) >= 6 else "N/A"
                print(f"    {i:2d}. {year}년 {mon}월 ({month}): {total:,.2f}원 ({percentage:.2f}%)")
            else:
                print(f"    {i:2d}. (결측값): {total:,.2f}원 ({percentage:.2f}%)")
    
    # 유효시작월별 통계
    valid_col_sums = col_sums[col_sums.index.notna()]
    if len(valid_col_sums) > 0:
        print(f"\n  - 유효시작월별 수익 통계:")
        print(f"    • 평균: {valid_col_sums.mean():,.2f}원")
        print(f"    • 중앙값: {valid_col_sums.median():,.2f}원")
        print(f"    • 최대값: {valid_col_sums.max():,.2f}원")
        print(f"    • 최소값: {valid_col_sums.min():,.2f}원")
        print(f"    • 표준편차: {valid_col_sums.std():,.2f}원")
    
    # ========================================
    # 4단계: 전체 통계 및 분석
    # ========================================
    print(f"\n[4단계] 전체 통계 및 분석")
    print("-" * 70)
    
    # 전체 합계 - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_pivot = pivot_df[numeric_cols]
        total_sum = numeric_pivot.sum().sum()
    else:
        total_sum = pivot_df.sum().sum()
    print(f"  - 전체 수익 합계: {total_sum:,.2f}원")
    
    # 양수/음수/0 분포 - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_pivot = pivot_df[numeric_cols]
        positive_count = (numeric_pivot > 0).sum().sum()
        negative_count = (numeric_pivot < 0).sum().sum()
        zero_count = (numeric_pivot == 0).sum().sum()
        total_cells = len(numeric_pivot) * len(numeric_pivot.columns)
    else:
        positive_count = (pivot_df > 0).sum().sum()
        negative_count = (pivot_df < 0).sum().sum()
        zero_count = (pivot_df == 0).sum().sum()
        total_cells = len(pivot_df) * len(pivot_df.columns)
    
    print(f"\n  - 셀 값 분포:")
    print(f"    • 양수: {positive_count:,}개 ({positive_count/total_cells*100:.2f}%)")
    print(f"    • 음수: {negative_count:,}개 ({negative_count/total_cells*100:.2f}%)")
    print(f"    • 0: {zero_count:,}개 ({zero_count/total_cells*100:.2f}%)")
    
    # 양수/음수 합계 - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        numeric_pivot = pivot_df[numeric_cols]
        positive_sum = numeric_pivot[numeric_pivot > 0].sum().sum()
        negative_sum = numeric_pivot[numeric_pivot < 0].sum().sum()
    else:
        positive_sum = pivot_df[pivot_df > 0].sum().sum()
        negative_sum = pivot_df[pivot_df < 0].sum().sum()
    
    print(f"\n  - 수익 합계 분포:")
    print(f"    • 양수 합계: {positive_sum:,.2f}원")
    print(f"    • 음수 합계: {negative_sum:,.2f}원")
    print(f"    • 순수익: {total_sum:,.2f}원")
    
    if positive_sum != 0:
        net_margin = (total_sum / positive_sum) * 100
        print(f"    • 순수익률: {net_margin:.2f}%")
    
    # 상위/하위 셀 값 - 숫자 컬럼만 선택
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        all_values = pivot_df[numeric_cols].values.flatten()
    else:
        all_values = pivot_df.values.flatten()
    all_values = all_values[all_values != 0]  # 0 제외
    
    if len(all_values) > 0:
        print(f"\n  - 셀 값 통계 (0 제외):")
        print(f"    • 평균: {all_values.mean():,.2f}원")
        print(f"    • 중앙값: {np.median(all_values):,.2f}원")
        print(f"    • 최대값: {all_values.max():,.2f}원")
        print(f"    • 최소값: {all_values.min():,.2f}원")
        print(f"    • 표준편차: {all_values.std():,.2f}원")
    
    # ========================================
    # 5단계: 샘플 데이터 출력
    # ========================================
    print(f"\n[5단계] 샘플 데이터")
    print("-" * 70)
    
    # 상위 5개 상품명과 상위 5개 유효시작월의 교차표
    top_products = row_sums.head(5).index.tolist()
    top_months = col_sums.head(5).index.tolist()
    
    print(f"  - 상위 5개 상품명 × 상위 5개 유효시작월 교차표:")
    # 숫자 컬럼만 선택하여 샘플 출력 (MVNO상품구분 제외)
    numeric_cols = pivot_df.select_dtypes(include=[np.number]).columns
    # top_months에서 숫자 컬럼만 필터링
    numeric_top_months = [m for m in top_months if m in numeric_cols]
    if len(numeric_top_months) > 0:
        sample_pivot = pivot_df.loc[top_products, numeric_top_months]
    else:
        sample_pivot = pivot_df.loc[top_products, top_months]
    
    # pandas 출력 옵션 설정
    with pd.option_context('display.max_rows', None, 
                           'display.max_columns', None,
                           'display.width', None,
                           'display.float_format', '{:,.2f}'.format):
        print(sample_pivot.to_string())
    
    print(f"\n  ✅ {pivot_df_name} 요약 정보 출력 완료!")

print("\n" + "="*70)
print("✅ 모든 피봇 테이블 요약 정보 출력 완료!")
print("="*70)



📊 피봇 테이블 요약 정보

📋 LGU_202507_PIVOT (2025년 07월)

[1단계] 기본 정보
----------------------------------------------------------------------
  - 데이터프레임명: LGU_202507_PIVOT
  - 행 수: 133행 (MVNO상품명)
  - 열 수: 53열 (유효시작월)
  - 총 셀 수: 7,049개
  - 메모리 사용량: 0.08 MB
  - 0이 아닌 셀 수: 1,742개 (25.19%)
  - 0인 셀 수: 5,174개 (74.81%)

[2단계] 상품명별 수익 통계
----------------------------------------------------------------------
  - 상위 10개 상품명 (수익 기준):
     1. [INS][평생할인]데이터 안심 10GB+(USIM): 41,177,352.77원 (-14.02%)
     2. [INS]데이터 안심 6GB+(USIM): 13,483,809.41원 (-4.59%)
     3. [INS]올프리 5GB+(USIM): 9,981,126.86원 (-3.40%)
     4. [INS]데이터 안심 10GB+(USIM): 4,559,857.86원 (-1.55%)
     5. [INS][평생할인]실속 300분+5GB(USIM): 4,325,898.10원 (-1.47%)
     6. GL수납 정산상품: 3,215,833.30원 (-1.10%)
     7. [INS]실속 150분+1GB(USIM): 2,269,369.47원 (-0.77%)
     8. [INS] [24개월]인스 유심 11GB+(알뜰폰플러스): 2,267,604.35원 (-0.77%)
     9. [INS]데이터 안심 1GB+(USIM): 2,138,617.46원 (-0.73%)
    10. [INS]실속 300분+5GB(USIM): 1,917,714.14원 (-0.65%)

  - 하위 5개 상품명 (수익 기준)

## 4️⃣ 데이터프레임 요약 정보 출력


In [58]:
print("\n" + "="*70)
print("📊 데이터프레임 요약 정보")
print("="*70)

for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n{'='*70}")
    print(f"📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월)")
    print(f"{'='*70}")
    
    # 기본 정보
    print(f"\n📊 기본 정보:")
    print(f"  - 데이터프레임명: {df_name}")
    print(f"  - 파일 경로: {df_info['file_path']}")
    print(f"  - 인코딩: {df_info['encoding']}")
    print(f"  - 행 수: {len(df):,}행")
    print(f"  - 열 수: {len(df.columns)}열")
    
    # 파일 크기 및 메모리 사용량
    file_size = df_info['file_path'].stat().st_size / 1024 / 1024
    memory_usage = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"  - 파일 크기: {file_size:.2f} MB")
    print(f"  - 메모리 사용량: {memory_usage:.2f} MB")
    
    # 컬럼 정보 (표 형태로 출력)
    print(f"\n📋 컬럼 정보:")
    print(f"  - 총 컬럼 수: {len(df.columns)}개")
    
    # 컬럼 정보를 표 형태로 생성
    columns_info = []
    for idx, col in enumerate(df.columns, 1):
        dtype = str(df[col].dtype)
        non_null_count = df[col].notna().sum()
        null_count = df[col].isnull().sum()
        null_percentage = (null_count / len(df)) * 100 if len(df) > 0 else 0
        
        columns_info.append({
            '번호': idx,
            '컬럼명': col,
            '데이터타입': dtype,
            '비결측값': f"{non_null_count:,}",
            '결측값': f"{null_count:,}",
            '결측률(%)': f"{null_percentage:.2f}"
        })
    
    df_columns_info = pd.DataFrame(columns_info)
    
    print(f"\n📊 컬럼 목록 (표 형태 - 전체 {len(df.columns)}개 컬럼):")
    
    # pandas 출력 옵션 설정 (모든 행 표시)
    with pd.option_context('display.max_rows', None, 
                           'display.max_columns', None,
                           'display.width', None,
                           'display.max_colwidth', 50):
        # 표 출력 (전체 컬럼 표시)
        print(df_columns_info.to_string(index=False))
    
    # 추가 정보
    print(f"\n  ℹ️ 총 {len(df.columns)}개 컬럼이 모두 표시되었습니다.")
    
    # 데이터 타입 정보
    print(f"\n🔢 데이터 타입:")
    dtype_counts = df.dtypes.value_counts()
    for dtype, count in dtype_counts.items():
        print(f"  - {dtype}: {count}개")
    
    # 결측값 정보
    print(f"\n⚠️ 결측값 정보:")
    missing_counts = df.isnull().sum()
    missing_cols = missing_counts[missing_counts > 0]
    if len(missing_cols) > 0:
        print(f"  - 결측값이 있는 컬럼: {len(missing_cols)}개")
        print(f"  - 결측값이 많은 상위 5개 컬럼:")
        for col, count in missing_cols.nlargest(5).items():
            percentage = (count / len(df)) * 100
            print(f"    • {col}: {count:,}개 ({percentage:.2f}%)")
    else:
        print(f"  - 결측값이 있는 컬럼 없음")
    
    # 선후불 구분 컬럼 유형값 출력
    print(f"\n💳 선후불 구분 컬럼 정보:")
    payment_cols = ['선후불구분', '선후불', '선후불구분명', '선후불명']
    payment_col = None
    
    for col in payment_cols:
        if col in df.columns:
            payment_col = col
            break
    
    if payment_col:
        print(f"  - 컬럼명: '{payment_col}'")
        value_counts = df[payment_col].value_counts()
        total_count = len(df)
        
        print(f"  - 유형값 및 개수:")
        for value, count in value_counts.items():
            percentage = (count / total_count) * 100
            if pd.isna(value):
                print(f"    • (결측값): {count:,}개 ({percentage:.2f}%)")
            else:
                print(f"    • {value}: {count:,}개 ({percentage:.2f}%)")
        
        print(f"  - 총 유형값 수: {len(value_counts)}개")
    else:
        print(f"  - ⚠️ 선후불 구분 컬럼을 찾을 수 없습니다.")
        print(f"    검색한 컬럼명: {', '.join(payment_cols)}")
    
    # 샘플 데이터 (상위 3행)
    print(f"\n👀 샘플 데이터 (상위 3행):")
    display_cols = df.columns.tolist()[:30]  # 처음 30개 컬럼만 표시
    print(df[display_cols].head(3).to_string())

print("\n" + "="*70)
print("✅ 모든 데이터프레임 요약 정보 출력 완료!")
print("="*70)



📊 데이터프레임 요약 정보

📋 LGU_202507 (2025년 07월)

📊 기본 정보:
  - 데이터프레임명: LGU_202507
  - 파일 경로: csv\202507_SS001344_ENTR_BY_STACC_PTN_INS_001.csv
  - 인코딩: cp949
  - 행 수: 170,639행
  - 열 수: 114열
  - 파일 크기: 121.97 MB
  - 메모리 사용량: 327.24 MB

📋 컬럼 정보:
  - 총 컬럼 수: 114개

📊 컬럼 목록 (표 형태 - 전체 114개 컬럼):
 번호                                       컬럼명          데이터타입    비결측값     결측값 결측률(%)
  1                                      정산년월          int64 170,639       0   0.00
  2                                      가입번호        float64 170,639       0   0.00
  3                                       고객명         object 170,639       0   0.00
  4                                  MVNO상품코드         object 170,639       0   0.00
  5                                   MVNO상품명         object 170,639       0   0.00
  6                                  MVNO상품구분         object 170,639       0   0.00
  7                                      마켓코드         object 170,639       0   0.00
  8                                       마

## 4-1️⃣ MVNO상품명 컬럼 정보 분석

각 데이터프레임의 MVNO상품명 컬럼에 대한 상세 분석을 수행합니다.

### 분석 내용
- MVNO상품명 컬럼 자동 감지
- 유형값(고유값) 목록 및 개수 (전체 표시)
- 각 상품명별 비율 및 통계 (표 형태)
- 결측값 정보
- 최다/최소 상품명 정보


In [59]:
print("\n" + "="*70)
print("📦 MVNO상품명 컬럼 정보 분석")
print("="*70)

# 각 데이터프레임별로 MVNO상품명 컬럼 분석
for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    print(f"\n{'='*70}")
    print(f"📋 {df_name} ({yyyymm[:4]}년 {yyyymm[4:]}월) - MVNO상품명 분석")
    print(f"{'='*70}")
    
    # ========================================
    # 1단계: MVNO상품명 컬럼 자동 감지
    # ========================================
    print(f"\n[1단계] MVNO상품명 컬럼 자동 감지")
    print("-" * 70)
    
    # 가능한 컬럼명 목록 (우선순위 순서)
    product_cols = ['MVNO상품명', '상품명', '요금제명', '개통요금제명', '현재요금제명']
    product_col = None
    
    # 컬럼명 자동 검색
    for col in product_cols:
        if col in df.columns:
            product_col = col
            print(f"  ✅ 컬럼 발견: '{col}'")
            break
    
    # 컬럼을 찾지 못한 경우
    if product_col is None:
        print(f"  ⚠️ MVNO상품명 컬럼을 찾을 수 없습니다.")
        print(f"     검색한 컬럼명: {', '.join(set(product_cols))}")
        print(f"     사용 가능한 컬럼 중 일부: {', '.join(df.columns.tolist()[:10])}")
        continue
    
    # ========================================
    # 2단계: 기본 통계 정보 수집
    # ========================================
    print(f"\n[2단계] 기본 통계 정보 수집")
    print("-" * 70)
    
    # 유형값(고유값) 개수 계산
    value_counts = df[product_col].value_counts()
    total_count = len(df)
    unique_count = len(value_counts)
    
    # 결측값 개수 계산
    missing_count = df[product_col].isnull().sum()
    non_missing_count = df[product_col].notna().sum()
    
    print(f"  - 컬럼명: '{product_col}'")
    print(f"  - 전체 행 수: {total_count:,}행")
    print(f"  - 총 유형값 수: {unique_count:,}개")
    print(f"  - 비결측값: {non_missing_count:,}행")
    print(f"  - 결측값: {missing_count:,}행 ({missing_count/total_count*100:.2f}%)")
    
    # ========================================
    # 3단계: 유형값별 통계 계산
    # ========================================
    print(f"\n[3단계] 유형값별 통계 계산")
    print("-" * 70)
    
    # 전체 유형값 사용 (제한 없음)
    print(f"  - 전체 {unique_count:,}개 유형값 분석:")
    
    # 표 형태로 출력하기 위한 데이터 준비 (전체 유형값)
    product_stats = []
    for i, (value, count) in enumerate(value_counts.items(), 1):
        percentage = (count / total_count) * 100
        
        # 결측값 처리
        if pd.isna(value):
            display_value = "(결측값)"
            product_stats.append({
                '순위': i,
                '상품명': display_value,
                '개수': count,
                '비율(%)': f"{percentage:.2f}"
            })
        else:
            # 전체 상품명 표시
            display_value = str(value)
            product_stats.append({
                '순위': i,
                '상품명': display_value,
                '개수': count,
                '비율(%)': f"{percentage:.2f}"
            })
    
    # ========================================
    # 4단계: 표 형태로 출력 (전체 유형값)
    # ========================================
    print(f"\n[4단계] 유형값 통계 표 출력 (전체)")
    print("-" * 70)
    
    if product_stats:
        df_product_stats = pd.DataFrame(product_stats)
        print(f"\n📊 전체 {unique_count:,}개 MVNO상품명 통계:")
        print(df_product_stats.to_string(index=False))
    
    # ========================================
    # 5단계: 추가 통계 정보
    # ========================================
    print(f"\n[5단계] 추가 통계 정보")
    print("-" * 70)
    
    # 가장 많은 상품명과 가장 적은 상품명
    if len(value_counts) > 0:
        max_product = value_counts.index[0]
        max_count = value_counts.iloc[0]
        max_percentage = (max_count / total_count) * 100
        
        min_product = value_counts.index[-1]
        min_count = value_counts.iloc[-1]
        min_percentage = (min_count / total_count) * 100
        
        print(f"  - 가장 많은 상품명:")
        max_display = str(max_product)[:60] + "..." if len(str(max_product)) > 60 else str(max_product)
        print(f"    '{max_display}'")
        print(f"    → {max_count:,}건 ({max_percentage:.2f}%)")
        
        print(f"\n  - 가장 적은 상품명:")
        min_display = str(min_product)[:60] + "..." if len(str(min_product)) > 60 else str(min_product)
        print(f"    '{min_display}'")
        print(f"    → {min_count:,}건 ({min_percentage:.2f}%)")
    
    print(f"\n{'='*70}")

print("\n" + "="*70)
print("✅ MVNO상품명 컬럼 정보 분석 완료!")
print("="*70)



📦 MVNO상품명 컬럼 정보 분석

📋 LGU_202507 (2025년 07월) - MVNO상품명 분석

[1단계] MVNO상품명 컬럼 자동 감지
----------------------------------------------------------------------
  ✅ 컬럼 발견: 'MVNO상품명'

[2단계] 기본 통계 정보 수집
----------------------------------------------------------------------
  - 컬럼명: 'MVNO상품명'
  - 전체 행 수: 170,639행
  - 총 유형값 수: 133개
  - 비결측값: 170,639행
  - 결측값: 0행 (0.00%)

[3단계] 유형값별 통계 계산
----------------------------------------------------------------------
  - 전체 133개 유형값 분석:

[4단계] 유형값 통계 표 출력 (전체)
----------------------------------------------------------------------

📊 전체 133개 MVNO상품명 통계:
 순위                              상품명    개수 비율(%)
  1                        번호이동 정산상품 57354 33.61
  2      [INS][24개월] 인스 유심 스트롱 11GB+ 14432  8.46
  3    [INS][평생할인]데이터 안심 10GB+(USIM) 11360  6.66
  4              [INS]인스 유심 스트롱 11G+  8226  4.82
  5            [INS] 인스 유심 스트롱 15GB+  8199  4.80
  6             [INS]인스 유심 올프리(7GB+)  7867  4.61
  7                       MNP채권 정산상품  6336  3.71
  8        [INS][모요

## 5️⃣ 데이터프레임 비교 분석 (선택사항)


In [60]:
if len(dataframes) > 1:
    print("\n" + "="*70)
    print("📊 데이터프레임 비교 분석")
    print("="*70)
    
    # 비교 테이블 생성
    comparison_data = []
    for yyyymm in sorted(dataframes.keys()):
        df_info = dataframes[yyyymm]
        df = df_info['df']
        file_size = df_info['file_path'].stat().st_size / 1024 / 1024
        memory_usage = df.memory_usage(deep=True).sum() / 1024 / 1024
        
        comparison_data.append({
            '연월': yyyymm,
            '데이터프레임명': df_info['name'],
            '행 수': len(df),
            '열 수': len(df.columns),
            '파일 크기 (MB)': round(file_size, 2),
            '메모리 사용량 (MB)': round(memory_usage, 2),
            '결측값 총계': df.isnull().sum().sum(),
            '중복 행 수': df.duplicated().sum()
        })
    
    df_comparison = pd.DataFrame(comparison_data)
    
    print("\n📋 데이터프레임 비교 테이블:")
    print(df_comparison.to_string(index=False))
    
    # 컬럼 비교
    print("\n\n📋 컬럼 비교:")
    all_columns = set()
    for yyyymm in sorted(dataframes.keys()):
        all_columns.update(dataframes[yyyymm]['df'].columns)
    
    print(f"  - 전체 고유 컬럼 수: {len(all_columns)}개")
    
    # 각 데이터프레임별 컬럼 수
    for yyyymm in sorted(dataframes.keys()):
        df_cols = set(dataframes[yyyymm]['df'].columns)
        print(f"  - {dataframes[yyyymm]['name']}: {len(df_cols)}개 컬럼")
    
    # 공통 컬럼
    common_columns = set(dataframes[sorted(dataframes.keys())[0]]['df'].columns)
    for yyyymm in sorted(dataframes.keys())[1:]:
        common_columns = common_columns.intersection(set(dataframes[yyyymm]['df'].columns))
    
    print(f"\n  - 공통 컬럼 수: {len(common_columns)}개")
    if len(common_columns) > 0 and len(common_columns) <= 20:
        print(f"  - 공통 컬럼 목록: {', '.join(sorted(common_columns))}")
    
    # 고유 컬럼 (각 데이터프레임에만 있는 컬럼)
    for yyyymm in sorted(dataframes.keys()):
        df_cols = set(dataframes[yyyymm]['df'].columns)
        unique_cols = df_cols - common_columns
        if len(unique_cols) > 0:
            print(f"\n  - {dataframes[yyyymm]['name']} 고유 컬럼 ({len(unique_cols)}개):")
            print(f"    {', '.join(sorted(unique_cols))}")
else:
    print("\n⚠️ 비교 분석을 위해서는 최소 2개 이상의 데이터프레임이 필요합니다.")



📊 데이터프레임 비교 분석

📋 데이터프레임 비교 테이블:
    연월    데이터프레임명    행 수  열 수  파일 크기 (MB)  메모리 사용량 (MB)  결측값 총계  중복 행 수
202507 LGU_202507 170639  114      121.97        327.24  306024    1269
202508 LGU_202508 161242  114      242.89        311.62  287766       0
202509 LGU_202509 152444  114      233.50        294.79  271116       0
202510 LGU_202510 168808  114      248.70        326.10  304286       0


📋 컬럼 비교:
  - 전체 고유 컬럼 수: 114개
  - LGU_202507: 114개 컬럼
  - LGU_202508: 114개 컬럼
  - LGU_202509: 114개 컬럼
  - LGU_202510: 114개 컬럼

  - 공통 컬럼 수: 114개


## 6️⃣ CSV 파일 저장

전처리된 데이터프레임 및 피봇 테이블을 `output` 폴더에 CSV 파일로 저장합니다.

**저장 파일**:
- 원본 데이터프레임: `output/LGU_YYYYMM_processed.csv` (전처리된 원본 데이터)
- 피봇 테이블: `output/LGU_YYYYMM_pivot.csv` (유효시작월 × MVNO상품명 피봇 테이블)

**저장 옵션**:
- 인코딩: UTF-8 with BOM (Excel 호환성)
- 인덱스: 저장하지 않음 (원본 데이터프레임), 저장함 (피봇 테이블)


In [61]:
# OUTPUT 디렉토리 설정 및 생성
OUTPUT_DIR = BASE_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

print("\n" + "="*70)
print("💾 CSV 파일 저장")
print("="*70)
print(f"📂 저장 디렉토리: {OUTPUT_DIR.absolute()}")

# 저장된 파일 목록
saved_files = []

# ========================================
# 1단계: 원본 데이터프레임 저장
# ========================================
print("\n[1단계] 원본 데이터프레임 저장")
print("-" * 70)

for yyyymm in sorted(dataframes.keys()):
    df_info = dataframes[yyyymm]
    df = df_info['df']
    df_name = df_info['name']
    
    # 저장 파일명: LGU_YYYYMM_processed.csv
    output_filename = f"{df_name}_processed.csv"
    output_path = OUTPUT_DIR / output_filename
    
    try:
        # CSV 파일로 저장 (UTF-8 with BOM for Excel compatibility)
        df.to_csv(
            output_path,
            index=False,  # 인덱스 저장하지 않음
            encoding='utf-8-sig',  # UTF-8 with BOM (Excel 호환성)
            na_rep=''  # 결측값은 빈 문자열로 저장
        )
        
        # 파일 크기 확인
        file_size = output_path.stat().st_size / 1024 / 1024
        
        print(f"  ✅ {df_name} → {output_filename}")
        print(f"     저장 경로: {output_path}")
        print(f"     파일 크기: {file_size:.2f} MB")
        print(f"     데이터 크기: {len(df):,}행 × {len(df.columns)}열")
        
        saved_files.append({
            'type': '원본 데이터프레임',
            'filename': output_filename,
            'path': output_path,
            'size_mb': file_size,
            'rows': len(df),
            'cols': len(df.columns)
        })
        
    except Exception as e:
        print(f"  ❌ {df_name} 저장 실패: {str(e)[:200]}")

# ========================================
# 2단계: 피봇 테이블 저장
# ========================================
print("\n[2단계] 피봇 테이블 저장")
print("-" * 70)

if 'pivot_dataframes' in globals() and len(pivot_dataframes) > 0:
    for yyyymm in sorted(pivot_dataframes.keys()):
        pivot_info = pivot_dataframes[yyyymm]
        pivot_df = pivot_info['df']
        pivot_df_name = pivot_info['name']
        
        # 저장 파일명: LGU_YYYYMM_pivot.csv
        output_filename = f"{pivot_df_name}.csv"
        output_path = OUTPUT_DIR / output_filename
        
        try:
            # CSV 파일로 저장 (UTF-8 with BOM for Excel compatibility)
            pivot_df.to_csv(
                output_path,
                index=True,  # 인덱스 저장 (MVNO상품명)
                encoding='utf-8-sig',  # UTF-8 with BOM (Excel 호환성)
                na_rep=''  # 결측값은 빈 문자열로 저장
            )
            
            # 파일 크기 확인
            file_size = output_path.stat().st_size / 1024 / 1024
            
            print(f"  ✅ {pivot_df_name} → {output_filename}")
            print(f"     저장 경로: {output_path}")
            print(f"     파일 크기: {file_size:.2f} MB")
            print(f"     데이터 크기: {len(pivot_df):,}행 × {len(pivot_df.columns)}열")
            
            saved_files.append({
                'type': '피봇 테이블',
                'filename': output_filename,
                'path': output_path,
                'size_mb': file_size,
                'rows': len(pivot_df),
                'cols': len(pivot_df.columns)
            })
            
        except Exception as e:
            print(f"  ❌ {pivot_df_name} 저장 실패: {str(e)[:200]}")
else:
    print("  ⚠️ 저장할 피봇 테이블이 없습니다.")

# ========================================
# 3단계: 저장 요약 정보
# ========================================
print("\n[3단계] 저장 요약 정보")
print("-" * 70)

if len(saved_files) > 0:
    print(f"  - 총 저장된 파일 수: {len(saved_files)}개")
    
    # 파일 타입별 통계
    df_files = [f for f in saved_files if f['type'] == '원본 데이터프레임']
    pivot_files = [f for f in saved_files if f['type'] == '피봇 테이블']
    
    print(f"  - 원본 데이터프레임: {len(df_files)}개")
    print(f"  - 피봇 테이블: {len(pivot_files)}개")
    
    # 전체 파일 크기 합계
    total_size = sum(f['size_mb'] for f in saved_files)
    print(f"  - 전체 파일 크기: {total_size:.2f} MB")
    
    print(f"\n  📋 저장된 파일 목록:")
    for i, file_info in enumerate(saved_files, 1):
        print(f"    {i}. [{file_info['type']}] {file_info['filename']}")
        print(f"       크기: {file_info['size_mb']:.2f} MB, "
              f"데이터: {file_info['rows']:,}행 × {file_info['cols']}열")
    
    print(f"\n  ✅ 모든 파일이 '{OUTPUT_DIR.absolute()}' 폴더에 저장되었습니다.")
else:
    print("  ⚠️ 저장된 파일이 없습니다.")

print("\n" + "="*70)
print("✅ CSV 파일 저장 완료!")
print("="*70)



💾 CSV 파일 저장
📂 저장 디렉토리: d:\Dev\Python_Data_Agent\output

[1단계] 원본 데이터프레임 저장
----------------------------------------------------------------------
  ✅ LGU_202507 → LGU_202507_processed.csv
     저장 경로: output\LGU_202507_processed.csv
     파일 크기: 100.49 MB
     데이터 크기: 170,639행 × 114열
  ✅ LGU_202508 → LGU_202508_processed.csv
     저장 경로: output\LGU_202508_processed.csv
     파일 크기: 107.81 MB
     데이터 크기: 161,242행 × 114열
  ✅ LGU_202509 → LGU_202509_processed.csv
     저장 경로: output\LGU_202509_processed.csv
     파일 크기: 102.29 MB
     데이터 크기: 152,444행 × 114열
  ✅ LGU_202510 → LGU_202510_processed.csv
     저장 경로: output\LGU_202510_processed.csv
     파일 크기: 112.38 MB
     데이터 크기: 168,808행 × 114열

[2단계] 피봇 테이블 저장
----------------------------------------------------------------------
  ✅ LGU_202507_PIVOT → LGU_202507_PIVOT.csv
     저장 경로: output\LGU_202507_PIVOT.csv
     파일 크기: 0.05 MB
     데이터 크기: 133행 × 53열
  ✅ LGU_202508_PIVOT → LGU_202508_PIVOT.csv
     저장 경로: output\LGU_202508_PIVOT.csv
     

## 🎉 처리 완료

### 처리 요약
1. ✅ 원시데이터 파일 탐색 완료
2. ✅ 모든 파일을 데이터프레임으로 로드 완료
3. ✅ 유효시작월 컬럼 추가 완료
4. ✅ 수익 컬럼 추가 완료
5. ✅ MVNO상품구분 컬럼 추가 완료
6. ✅ 선불 항목 필터링 완료
7. ✅ 피봇 테이블 생성 완료
8. ✅ 피봇 테이블 요약 정보 출력 완료
9. ✅ 데이터프레임 요약 정보 출력 완료
10. ✅ MVNO상품명 컬럼 상세 분석 완료
11. ✅ CSV 파일 저장 완료

### 사용 가능한 데이터프레임
다음과 같은 이름으로 데이터프레임이 생성되었습니다:

- **원본 데이터프레임**: `LGU_YYYYMM` 형식 (예: `LGU_202507`, `LGU_202510`)
- **피봇 테이블**: `LGU_YYYYMM_PIVOT` 형식 (예: `LGU_202507_PIVOT`, `LGU_202510_PIVOT`)

### 저장된 파일
전처리된 데이터는 `output` 폴더에 저장되었습니다:

- **원본 데이터프레임**: `output/LGU_YYYYMM_processed.csv`
- **피봇 테이블**: `output/LGU_YYYYMM_PIVOT.csv`

### 다음 단계
- 저장된 CSV 파일을 Excel이나 다른 분석 도구에서 활용
- 다른 노트북에서 로드된 데이터프레임 활용
- 추가 데이터 분석 및 시각화 수행
